# Kira LLM - 100% Human Data Training Pipeline
### Strict "No Other Model" Rule: Zero Synthetic Tokens, Zero Distillation, Zero AI Filtering

This notebook trains a complete ~15M–30M parameter transformer from scratch:
1. **Pre-training**: 100% human-authored text (English Wikipedia & Project Gutenberg).
2. **Custom Tokenizer**: Byte-Level BPE trained from scratch (vocab size = 8,192). No pre-trained GPT tokenizers.
3. **Pre-training Execution**: Fast PyTorch SDPA (FlashAttention) with mixed precision on free Colab T4 GPU.
4. **Supervised Fine-Tuning (SFT)**: 100% human chat datasets (`databricks-dolly-15k` & `OpenAssistant/oasst1`).
5. **Interactive Chat**: Test your newly trained model directly.

## Step 1: Check GPU & Install Dependencies

In [ ]:
!nvidia-smi
!pip install datasets tokenizers numpy tqdm torch
!git clone -b fix/server-auth-hardening https://github.com/harctik/kira.ai.git
%cd kira.ai


## Step 1b: Mount Google Drive (so checkpoints survive a disconnect)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/kira


## Step 2: Download Human Text & Train Custom BPE Tokenizer
Downloads 50,000 Wikipedia articles, trains a custom 8,192 vocab BPE tokenizer, and creates `train.bin` & `val.bin`.

In [ ]:
!python prepare_data.py --docs-per-source 50000 --vocab-size 8192 --output-dir data/kira

## Step 3: Pre-train Kira (15M Parameter Model)
Uses FlashAttention (SDPA) + Mixed Precision (fp16/bf16) on the Colab T4 GPU.

In [ ]:
!python train.py --data-dir data/kira --out-dir out --model-size 15m --batch-size 32 --gradient-accumulation-steps 4 --max-iters 5000

In [ ]:
!cp -r out data/kira/tokenizer.json data/kira/meta.pkl /content/drive/MyDrive/kira/


## Step 4: Prepare Human Chat Data (SFT)
Packs Databricks Dolly-15k and OpenAssistant oasst1 using the custom tokenizer.

In [ ]:
!python prepare_sft_data.py --tokenizer-path data/kira/tokenizer.json --output-dir data/kira_sft

## Step 5: Fine-Tune on Human Dialogue

In [ ]:
!python train.py --data-dir data/kira_sft --out-dir out_sft --model-size 15m --batch-size 16 --gradient-accumulation-steps 4 --max-iters 1500 --learning-rate 2e-4

In [ ]:
!mkdir -p /content/drive/MyDrive/kira/out_sft && cp out_sft/ckpt.pt /content/drive/MyDrive/kira/out_sft/


## Step 6: Test & Chat with Kira!

In [ ]:
# Run in interactive chat mode
!python chat.py --ckpt out_sft/ckpt.pt --tokenizer data/kira/tokenizer.json --mode chat